# build-raw-tissdiss-v2
Per-sample QC → concat → normalize → UMAP. Input h5 files come from the hisepy cache (see `02-download-tissdiss`); metadata and labels from `../reference/`.

In [ ]:
import glob, re, os
import numpy as np
import pandas as pd
import hisepy as hp
import scanpy as sc
import anndata as ad
import scanpy.external as sce
from scipy.stats import median_abs_deviation
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

RAW_OBJECT_PATH = Path('../data/raw-all-tissdiss.h5ad')
assert not RAW_OBJECT_PATH.exists(), f'{RAW_OBJECT_PATH} already exists; move it aside to rebuild'

sc.settings.verbosity = 1

In [ ]:
def is_outlier(adata, metric, nmads=5):
  M = adata.obs[metric]
  return (M < np.median(M) - nmads * median_abs_deviation(M)) | (M > np.median(M) + nmads * median_abs_deviation(M))


def process_sample(fname):
  path        = h5_paths[fname]
  source_file = fname.replace('_sample_filtered_feature_bc_matrix.h5', '')

  adata = sc.read_10x_h5(path)
  adata.var_names_make_unique()
  adata.obs['source_file']      = source_file
  adata.obs['original_barcode'] = adata.obs.index.astype(str)

  adata.var['mt'] = adata.var_names.str.startswith('MT-')
  sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=[20], log1p=True, inplace=True)

  adata.obs['outlier'] = (
      is_outlier(adata, 'log1p_total_counts')
      | is_outlier(adata, 'log1p_n_genes_by_counts')
      | is_outlier(adata, 'pct_counts_in_top_20_genes')
  )

  record = {
      'source_file':        source_file,
      'n_cells_raw':        adata.n_obs,
      'n_outliers_flagged': int(adata.obs['outlier'].sum()),
      'median_n_genes':     int(np.median(adata.obs['n_genes_by_counts'])),
      'median_n_counts':    int(np.median(adata.obs['total_counts'])),
      'median_pct_mt':      round(float(np.median(adata.obs['pct_counts_mt'])), 2),
  }

  return adata, record

In [ ]:
metadata = pd.read_csv('../reference/updated_metadata.csv')
pcd_store_files = [os.path.basename(p) for p in metadata[metadata['flag'].isna()]['file_path']]

store = hp.list_files_in_project_store('PCD_Spatial')
store['basename'] = store['name'].fillna('').map(os.path.basename)
matched = store[store['basename'].isin(pcd_store_files)]
assert not matched['basename'].duplicated().any(), 'Duplicate basenames in the store for wanted files'

h5_paths = {os.path.basename(path): path for path in hp.cache_files(file_ids=matched['id'].tolist())}
missing = sorted(set(pcd_store_files) - set(h5_paths))
assert not missing, f'No local file for: {missing}'

In [ ]:
with ThreadPoolExecutor(max_workers=24) as ex:
    results = list(ex.map(process_sample, pcd_store_files))

adata_list = [r[0] for r in results]
qc_records = [r[1] for r in results]

for rec in qc_records:
    print(f"{rec['source_file']}: {rec['n_cells_raw']} cells "
          f"({rec['n_outliers_flagged']} outliers flagged ")

qc_df = pd.DataFrame(qc_records)
print(f"\nTotal: {qc_df['n_cells_raw'].sum()} raw | {qc_df['n_outliers_flagged'].sum()} Outliers")

In [ ]:
qc_df = qc_df.merge(metadata, on='source_file', how='left', suffixes=('', '_meta'))

meta_cols = [c for c in metadata.columns if c not in ('source_file', 'TIS_ID', 'project')]
for adata, (_, row) in zip(adata_list, qc_df.iterrows()):
  for col in meta_cols:
      adata.obs[col] = row[col]

In [ ]:
qc_df.to_csv('../reference/qc-summary.csv', index=False)

In [ ]:
adata_merged = ad.concat(adata_list, join='outer')
adata_merged.var_names_make_unique()
adata_merged.obs_names_make_unique()
adata_merged.raw = adata_merged
adata_merged

In [ ]:
# build label map from old object — keyed on original_barcode (pre-uniquify suffix)
old_labels = pd.read_parquet('../reference/old_labels.parquet')['manual_map_leiden'].to_dict()
adata_merged.obs['old_labels'] = adata_merged.obs['original_barcode'].map(old_labels)

matched   = adata_merged.obs['old_labels'].notna().sum()
unmatched = adata_merged.obs['old_labels'].isna().sum()
print(f"Matched: {matched} | Unmatched (new cells): {unmatched}")

In [ ]:
os.makedirs(RAW_OBJECT_PATH.parent, exist_ok=True)
adata_merged.write(RAW_OBJECT_PATH)

In [ ]:
adata_merged